###Importar rutas

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

###Ingestion desde la capa Silver

In [0]:
#movie_df = spark.read.parquet(f"{silver_folder_path}/movies")
movie_df = spark.read.table("movie_silver.movies")\
                        .filter(f"file_date = '{v_file_date}'")

#country_df = spark.read.parquet(f"{silver_folder_path}/country")
country_df = spark.read.table("movie_silver.countries")

#prod_country_df = spark.read.parquet(f"{silver_folder_path}/production_country")
prod_country_df = spark.read.table("movie_silver.productions_countries")\
                            .filter(f"file_date = '{v_file_date}'")

#movie_comp_df = spark.read.parquet(f"{silver_folder_path}/movie_company")
movie_comp_df = spark.read.table("movie_silver.movies_companies")\
                            .filter(f"file_date = '{v_file_date}'")

#prod_comp_df = spark.read.parquet(f"{silver_folder_path}/productions_company")
prod_comp_df = spark.read.table("movie_silver.productions_companies")\
                            .filter(f"file_date = '{v_file_date}'")

###Join country y production_country

In [0]:
country_prod_country = country_df.join(prod_country_df,
                                       country_df.country_id == prod_country_df.country_id,
                                       "inner")\
                                    .select(country_df.country_name, country_df.country_id, prod_country_df.movie_id)

###Join movie_company y production_company

In [0]:
movie_comp_prod_df = prod_comp_df.join(movie_comp_df,
                                       prod_comp_df.company_id == movie_comp_df.company_id,
                                       "inner")\
                                    .select(prod_comp_df.company_name, prod_comp_df.company_id, movie_comp_df.movie_id)

###Join final

In [0]:
movie_joined_df = movie_df.join(country_prod_country,
                                movie_df.movie_id == country_prod_country.movie_id, "inner")\
                            .join(movie_comp_prod_df,
                                  movie_df.movie_id == movie_comp_prod_df.movie_id, "inner")

### filter, order, created_date y select

In [0]:
from pyspark.sql.functions import lit

In [0]:
final_df = movie_joined_df.filter(movie_df.year_release_date >= 2010)\
                            .select(movie_comp_prod_df.movie_id,
                                    prod_comp_df.company_id,
                                    country_df.country_id,
                                    "title",
                                    "budget",
                                    "revenue",
                                    "duration_time",
                                    "release_date",
                                    "country_name",
                                    "company_name")\
                            .orderBy("title")\
                            .withColumn("created_date", lit(v_file_date))


###Almacenar en capa Gold

In [0]:
#final_df.write\
#        .mode("overwrite")\
#        .parquet(f"{gold_folder_path}/results_country_prod_company")

In [0]:
%skip
dupe_prevent("movie_gold", "results_country_prod_company", "created_date", v_file_date)

In [0]:
condition_merge = 'tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.company_id = src.company_id AND tgt.created_date = src.created_date'

incremental_merge("movie_gold", "results_country_prod_company", final_df, condition_merge, "created_date")

#final_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_country_prod_company")

In [0]:
%sql
SELECT *
FROM movie_gold.results_country_prod_company
--WHERE created_date = '2024-12-23';

company_id,movie_id,country_id,title,budget,revenue,duration_time,release_date,country_name,company_name,created_date
75278,301325,214,#Horror,1500000.0,1875000.0,90,2015-11-20,United States of America,Lowland Pictures,2024-12-30
75277,301325,214,#Horror,1500000.0,1875000.0,90,2015-11-20,United States of America,AST Studios,2024-12-30
85248,433715,214,8 Days,2000000.0,2600000.0,90,2014-06-15,United States of America,After Eden Pictures,2024-12-30
68117,343795,214,90 Minutes in Heaven,5000000.0,4842699.0,121,2015-09-11,United States of America,Giving Films,2024-12-30
53656,241239,128,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United Arab Emirates,Old Bull Pictures,2024-12-30
41077,241239,128,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United Arab Emirates,A24,2024-12-30
53656,241239,214,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United States of America,Old Bull Pictures,2024-12-30
41077,241239,214,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United States of America,A24,2024-12-30
40107,169917,214,A Walk Among the Tombstones,2.8E7,5.31816E7,113,2014-09-18,United States of America,Da Vinci Media Ventures,2024-12-30
40106,169917,214,A Walk Among the Tombstones,2.8E7,5.31816E7,113,2014-09-18,United States of America,Free State Pictures,2024-12-30


In [0]:
%sql
SELECT created_date, COUNT(1)
FROM movie_gold.results_country_prod_company
GROUP BY created_date;

created_date,count(1)
2024-12-16,86
2024-12-23,171
2024-12-30,581
